In [ ]:
import numpy as np
import pandas as pd 

from sklearn.preprocessing import LabelEncoder

import os
import missingno as msno
import warnings
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

warnings.filterwarnings('ignore')

In [ ]:
ROOT_DIR = 'C:/Users/jfurs/@Python/OpenClassrooms/DS/P7/data/'

# UTILS

In [ ]:
def label_encoder(df, nan_as_category=True):
    """Encode categorical columns with LabelEncoder"""
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in cat_cols:
        df[col] = df[col].fillna('NaN') if nan_as_category else df[col]
        df[col] = LabelEncoder().fit_transform(df[col])
    return df, cat_cols

ANOMALY_VALUE = 365243

In [ ]:
ANOMALY_VALUE = 365243

def label_encoder(df, nan_as_category=True):
    """Encode categorical columns with LabelEncoder"""
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in cat_cols:
        df[col] = df[col].fillna('NaN') if nan_as_category else df[col]
        df[col] = LabelEncoder().fit_transform(df[col])
    return df, cat_cols

In [ ]:
# ------------------------
# Bureau + Bureau Balance
# ------------------------

def bureau_and_balance_light(num_rows=None, bureau_file=None, balance_file=None, output_file=None):
    bureau = pd.read_csv(bureau_file, nrows=num_rows)
    bb = pd.read_csv(balance_file, nrows=num_rows)
    
    bb, _ = label_encoder(bb)
    bureau, _ = label_encoder(bureau)
    
    # Aggregate bureau_balance
    bb_agg = bb.groupby('SK_ID_BUREAU').agg({
    'MONTHS_BALANCE': ['mean', 'size']
    })

    # Flatten MultiIndex columns
    bb_agg.columns = ['BB_' + '_'.join(col).upper() for col in bb_agg.columns]
    bb_agg = bb_agg.reset_index()  # SK_ID_BUREAU devient colonne normale
    bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
    
    # Aggregate numeric bureau features
    num_agg = {
        'DAYS_CREDIT': ['min', 'max', 'mean'],
        'DAYS_CREDIT_ENDDATE': ['min', 'max', 'mean'],
        'AMT_CREDIT_SUM': ['mean', 'sum'],
        'AMT_CREDIT_SUM_DEBT': ['mean', 'sum'],
        'AMT_CREDIT_SUM_OVERDUE': ['mean'],
        'AMT_CREDIT_SUM_LIMIT': ['mean', 'sum'],
        'AMT_ANNUITY': ['mean'],
        'CNT_CREDIT_PROLONG': ['sum']
    }
    
    bureau_agg = bureau.groupby('SK_ID_CURR').agg(num_agg)
    bureau_agg.columns = ['BURO_' + '_'.join(col).upper() for col in bureau_agg.columns]
    
    # Active credits only
    active = bureau[bureau['CREDIT_ACTIVE'] == 1]
    active_agg = active.groupby('SK_ID_CURR').agg(num_agg)
    active_agg.columns = ['ACTIVE_' + '_'.join(col).upper() for col in active_agg.columns]
    bureau_agg = bureau_agg.join(active_agg, how='left')
    
    # Closed credits only
    closed = bureau[bureau['CREDIT_ACTIVE'] == 0]
    closed_agg = closed.groupby('SK_ID_CURR').agg(num_agg)
    closed_agg.columns = ['CLOSED_' + '_'.join(col).upper() for col in closed_agg.columns]
    bureau_agg = bureau_agg.join(closed_agg, how='left')

    bureau_agg.to_parquet(output_file, index=True)

bureau_and_balance_light(bureau_file=f'{ROOT_DIR}bureau.csv',
                         balance_file=f'{ROOT_DIR}bureau_balance.csv',
                         output_file=f'{ROOT_DIR}bureau_agg.parquet')

In [ ]:
# ------------------------
# Previous Applications
# ------------------------

def previous_applications_light(num_rows=None, prev_file=None, output_file=None):
    prev = pd.read_csv(prev_file, nrows=num_rows)
    prev, _ = label_encoder(prev)
    
    # Replace anomaly
    for col in ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION',
                'DAYS_LAST_DUE', 'DAYS_TERMINATION']:
        prev[col].replace(ANOMALY_VALUE, np.nan, inplace=True)
    
    prev['APP_CREDIT_PERC'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT']
    
    num_agg = {
        'AMT_ANNUITY': ['mean'],
        'AMT_APPLICATION': ['mean', 'sum'],
        'AMT_CREDIT': ['mean', 'sum'],
        'APP_CREDIT_PERC': ['mean', 'var'],
        'AMT_DOWN_PAYMENT': ['mean'],
        'AMT_GOODS_PRICE': ['mean'],
        'DAYS_DECISION': ['mean'],
        'CNT_PAYMENT': ['sum']
    }
    
    prev_agg = prev.groupby('SK_ID_CURR').agg(num_agg)
    prev_agg.columns = ['PREV_' + '_'.join(col).upper() for col in prev_agg.columns]
    
    prev_agg.to_parquet(output_file, index=True)

previous_applications_light(prev_file=f'{ROOT_DIR}previous_application.csv',
                            output_file=f'{ROOT_DIR}previous_agg.parquet')

In [ ]:
# ------------------------
# POS-CASH Balance
# ------------------------

def pos_cash_light(num_rows=None, pos_file=None, output_file=None):
    pos = pd.read_csv(pos_file, nrows=num_rows)
    pos, _ = label_encoder(pos)
    
    pos_agg = pos.groupby('SK_ID_CURR').agg({
        'MONTHS_BALANCE': ['mean'],
        'SK_DPD': ['max', 'mean'],
        'SK_DPD_DEF': ['max', 'mean']
    })
    pos_agg.columns = ['POS_' + '_'.join(col).upper() for col in pos_agg.columns]
    
    pos_agg['POS_COUNT'] = pos.groupby('SK_ID_CURR').size()
    
    pos_agg.to_parquet(output_file, index=True)

pos_cash_light(pos_file=f'{ROOT_DIR}POS_CASH_balance.csv',
               output_file=f'{ROOT_DIR}pos_cash_agg.parquet')

In [ ]:
# ------------------------
# Installments Payments
# ------------------------

def installments_payments_light(num_rows=None, ins_file=None, output_file=None):
    ins = pd.read_csv(ins_file, nrows=num_rows)
    ins, _ = label_encoder(ins)
    
    ins['PAYMENT_PERC'] = ins['AMT_PAYMENT'] / ins['AMT_INSTALMENT']
    ins['DPD'] = (ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']).clip(lower=0)
    
    ins_agg = ins.groupby('SK_ID_CURR').agg({
        'DPD': ['max', 'mean', 'sum'],
        'PAYMENT_PERC': ['mean', 'sum', 'var'],
        'AMT_INSTALMENT': ['sum', 'mean'],
        'AMT_PAYMENT': ['sum', 'mean']
    })
    ins_agg.columns = ['INSTAL_' + '_'.join(col).upper() for col in ins_agg.columns]
    
    ins_agg['INSTAL_COUNT'] = ins.groupby('SK_ID_CURR').size()
    
    ins_agg.to_parquet(output_file, index=True)

installments_payments_light(ins_file=f'{ROOT_DIR}installments_payments.csv',
                           output_file=f'{ROOT_DIR}installments_agg.parquet')

In [ ]:
# ------------------------
# Credit Card Balance
# ------------------------

def credit_card_balance_light(num_rows=None, cc_file=None, output_file=None):
    cc = pd.read_csv(cc_file, nrows=num_rows)
    cc, _ = label_encoder(cc)
    cc.drop(['SK_ID_PREV'], axis=1, inplace=True)
    
    cc_agg = cc.groupby('SK_ID_CURR').agg({
        'AMT_BALANCE': ['mean', 'sum'],
        'AMT_CREDIT_LIMIT_ACTUAL': ['mean'],
        'AMT_DRAWINGS_CURRENT': ['mean', 'sum'],
        'AMT_PAYMENT_TOTAL_CURRENT': ['mean', 'sum'],
        'AMT_TOTAL_RECEIVABLE': ['mean', 'sum'],
        'CNT_DRAWINGS_CURRENT': ['mean', 'sum']
    })
    cc_agg.columns = ['CC_' + '_'.join(col).upper() for col in cc_agg.columns]
    
    cc_agg['CC_COUNT'] = cc.groupby('SK_ID_CURR').size()

    cc_agg.to_parquet(output_file, index=True)

credit_card_balance_light(cc_file=f'{ROOT_DIR}credit_card_balance.csv',
                         output_file=f'{ROOT_DIR}credit_card_agg.parquet')

In [ ]:
ROOT = r"C:\Users\jfurs\@Python\OpenClassrooms\DS\P7\data\ ".strip()

# List files available
pprint(os.listdir(ROOT))

['application_test.csv',
 'application_test_processed.parquet',
 'application_train.csv',
 'application_train_processed.parquet',
 'bureau.csv',
 'bureau_agg.parquet',
 'bureau_balance.csv',
 'complete_test.parquet',
 'credit_card_agg.parquet',
 'credit_card_balance.csv',
 'HomeCredit_columns_description.csv',
 'installments_agg.parquet',
 'installments_payments.csv',
 'pos_cash_agg.parquet',
 'POS_CASH_balance.csv',
 'previous_agg.parquet',
 'previous_application.csv',
 'sample_submission.csv',
 'X_res.parquet',
 'X_train_imp.parquet',
 'X_val.parquet',
 'y_res.parquet',
 'y_val.parquet']


In [ ]:
def preprocessing_train_test(file, output_file):
    df = pd.read_csv(file)
    
    # MAPPING BINAIRES

    df['FLAG_OWN_CAR'] = df['FLAG_OWN_CAR'].map({'Y': 1, 'N': 0})
    df['FLAG_OWN_REALTY'] = df['FLAG_OWN_REALTY'].map({'Y': 1, 'N': 0})
    df['CODE_GENDER'] = df['CODE_GENDER'].map({'M': 1, 'F': 0, 'XNA': np.nan})

    # DUMMIES

    cat_cols = ['NAME_CONTRACT_TYPE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 
                'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 
                'WEEKDAY_APPR_PROCESS_START', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 
                'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

    df = pd.get_dummies(df, columns=cat_cols, dummy_na=True)

# MAPPING ORDONNE

    edu_map = {
        'Lower secondary': 1,
        'Secondary / secondary special': 2,
        'Incomplete higher': 3,
        'Higher education': 4,
        'Academic degree': 5
    }
    df['NAME_EDUCATION_TYPE'] = df['NAME_EDUCATION_TYPE'].map(edu_map)

    # MONTANTS FINNANCIERS

    # Montant total du crédit rapporté au revenu annuel du client.
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

    # Annuité du crédit rapportée au revenu annuel du client.
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

    # Montant du crédit rapporté au prix du bien financé.
    df['CREDIT_GOODS_RATIO'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']

    # Annuité rapportée au montant total du crédit.
    df['ANNUITY_CREDIT_RATIO'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']

    # DATES

    df['AGE_YEARS'] = (-df['DAYS_BIRTH']) / 365
    df['EMPLOYED_YEARS'] = df['DAYS_EMPLOYED'].replace(ANOMALY_VALUE, np.nan) / -365

    df['EMPLOYED_TO_AGE_RATIO'] = df['EMPLOYED_YEARS'] / df['AGE_YEARS']

    df['REGISTRATION_YEARS'] = (-df['DAYS_REGISTRATION']) / 365
    df['ID_PUBLISH_YEARS'] = (-df['DAYS_ID_PUBLISH']) / 365
    df['PHONE_CHANGE_YEARS'] = (-df['DAYS_LAST_PHONE_CHANGE']) / 365

    df['RECENT_ID_CHANGE'] = (df['ID_PUBLISH_YEARS'] < 1).astype(int)
    df['RECENT_PHONE_CHANGE'] = (df['PHONE_CHANGE_YEARS'] < 1).astype(int)

    # DOCUMENTS

    doc_cols = [col for col in df.columns if 'FLAG_DOCUMENT' in col]
    df['DOCUMENT_COUNT'] = df[doc_cols].sum(axis=1)

    df.set_index('SK_ID_CURR', inplace=True, drop=True)

    df.to_parquet(output_file, index=True)

preprocessing_train_test(file=f'{ROOT_DIR}application_train.csv',
                         output_file=f'{ROOT_DIR}application_train_processed.parquet')

preprocessing_train_test(file=f'{ROOT_DIR}application_test.csv',
                         output_file=f'{ROOT_DIR}application_test_processed.parquet')

# APP_TRAIN FEATURE ENGINEERING MAISON

In [ ]:
app_train = pd.read_csv(ROOT + 'application_train.csv')
print('Training data shape: ', app_train.shape)
app_train.head()

Training data shape:  (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
app_test = pd.read_csv(ROOT + 'application_test.csv')
print('Testing data shape: ', app_test.shape)
app_test.head()

Testing data shape:  (48744, 121)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_train = app_train.copy()
df_test = app_test.copy()

### Variables binaires → mapping simple

In [ ]:
df_train[['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CODE_GENDER', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL']].head()

,FLAG_OWN_CAR,FLAG_OWN_REALTY,CODE_GENDER,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL
0,N,Y,M,1,1,0,1,1,0
1,N,N,F,1,1,0,1,1,0
2,Y,Y,M,1,1,1,1,1,0
3,N,Y,F,1,1,0,1,0,0
4,N,Y,M,1,1,0,1,0,0


In [ ]:
df_train["CODE_GENDER"].value_counts()

CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64

In [ ]:
df_train['FLAG_OWN_CAR'] = df_train['FLAG_OWN_CAR'].map({'Y': 1, 'N': 0})
df_train['FLAG_OWN_REALTY'] = df_train['FLAG_OWN_REALTY'].map({'Y': 1, 'N': 0})
df_train['CODE_GENDER'] = df_train['CODE_GENDER'].map({'M': 1, 'F': 0, 'XNA': np.nan})

### Variables catégorielles nominales → pd.get_dummies

In [ ]:
cat_cols = ['NAME_CONTRACT_TYPE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 
            'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 
            'WEEKDAY_APPR_PROCESS_START', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 
            'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

In [ ]:
for i in df_train[cat_cols]:
    print( f"{i}: {len(df_train[i].value_counts())}")

NAME_CONTRACT_TYPE: 2
NAME_TYPE_SUITE: 7
NAME_INCOME_TYPE: 8
NAME_FAMILY_STATUS: 6
NAME_HOUSING_TYPE: 6
OCCUPATION_TYPE: 18
ORGANIZATION_TYPE: 58
WEEKDAY_APPR_PROCESS_START: 7
FONDKAPREMONT_MODE: 4
HOUSETYPE_MODE: 3
WALLSMATERIAL_MODE: 7
EMERGENCYSTATE_MODE: 2


In [ ]:
df_train = pd.get_dummies(df_train, columns=cat_cols, dummy_na=True)

### Variables ordinales → mapping ordonné

In [ ]:
df_train['NAME_EDUCATION_TYPE'].value_counts()

NAME_EDUCATION_TYPE
Secondary / secondary special    218391
Higher education                  74863
Incomplete higher                 10277
Lower secondary                    3816
Academic degree                     164
Name: count, dtype: int64

In [ ]:
edu_map = {
    'Lower secondary': 1,
    'Secondary / secondary special': 2,
    'Incomplete higher': 3,
    'Higher education': 4,
    'Academic degree': 5
}
df_train['NAME_EDUCATION_TYPE'] = df_train['NAME_EDUCATION_TYPE'].map(edu_map)

In [ ]:
df_train[["REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY"]].value_counts()

REGION_RATING_CLIENT  REGION_RATING_CLIENT_W_CITY
2                     2                              225736
3                     3                               43860
1                     1                               32197
3                     2                                3748
2                     1                                1248
3                     1                                 722
Name: count, dtype: int64

### Variables numériques continues → transformations utiles : Montants financiers

In [ ]:
# Montant total du crédit rapporté au revenu annuel du client.
df_train['CREDIT_INCOME_RATIO'] = df_train['AMT_CREDIT'] / df_train['AMT_INCOME_TOTAL']

# Annuité du crédit rapportée au revenu annuel du client.
df_train['ANNUITY_INCOME_RATIO'] = df_train['AMT_ANNUITY'] / df_train['AMT_INCOME_TOTAL']

# Montant du crédit rapporté au prix du bien financé.
df_train['CREDIT_GOODS_RATIO'] = df_train['AMT_CREDIT'] / df_train['AMT_GOODS_PRICE']

# Annuité rapportée au montant total du crédit.
df_train['ANNUITY_CREDIT_RATIO'] = df_train['AMT_ANNUITY'] / df_train['AMT_CREDIT']

In [ ]:
df_train[["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_GOODS_RATIO", "ANNUITY_CREDIT_RATIO"]].head()

,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_GOODS_RATIO,ANNUITY_CREDIT_RATIO
0,2.007889,0.121978,1.158397,0.060749
1,4.790750,0.132217,1.145199,0.027598
2,2.000000,0.100000,1.000000,0.050000
3,2.316167,0.219900,1.052803,0.094941
4,4.222222,0.179963,1.000000,0.042623


### Variables de dates (en jours négatifs) → transformer en années

In [ ]:
df_train[["DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE"]].head()

,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,DAYS_LAST_PHONE_CHANGE
0,-9461,-637,-3648.0,-2120,-1134.0
1,-16765,-1188,-1186.0,-291,-828.0
2,-19046,-225,-4260.0,-2531,-815.0
3,-19005,-3039,-9833.0,-2437,-617.0
4,-19932,-3038,-4311.0,-3458,-1106.0


In [ ]:
df_train['AGE_YEARS'] = (-df_train['DAYS_BIRTH']) / 365
df_train['EMPLOYED_YEARS'] = df_train['DAYS_EMPLOYED'].replace(ANOMALY_VALUE, np.nan) / -365

df_train['EMPLOYED_TO_AGE_RATIO'] = df_train['EMPLOYED_YEARS'] / df_train['AGE_YEARS']

df_train['REGISTRATION_YEARS'] = (-df_train['DAYS_REGISTRATION']) / 365
df_train['ID_PUBLISH_YEARS'] = (-df_train['DAYS_ID_PUBLISH']) / 365
df_train['PHONE_CHANGE_YEARS'] = (-df_train['DAYS_LAST_PHONE_CHANGE']) / 365

df_train['RECENT_ID_CHANGE'] = (df_train['ID_PUBLISH_YEARS'] < 1).astype(int)
df_train['RECENT_PHONE_CHANGE'] = (df_train['PHONE_CHANGE_YEARS'] < 1).astype(int)

In [ ]:
df_train[["AGE_YEARS", "EMPLOYED_YEARS", "EMPLOYED_TO_AGE_RATIO", "REGISTRATION_YEARS", "ID_PUBLISH_YEARS", "PHONE_CHANGE_YEARS",
          "RECENT_ID_CHANGE", "RECENT_PHONE_CHANGE"]].head()

,AGE_YEARS,EMPLOYED_YEARS,EMPLOYED_TO_AGE_RATIO,REGISTRATION_YEARS,ID_PUBLISH_YEARS,PHONE_CHANGE_YEARS,RECENT_ID_CHANGE,RECENT_PHONE_CHANGE
0,25.920548,1.745205,0.067329,9.994521,5.808219,3.106849,0,0
1,45.931507,3.254795,0.070862,3.249315,0.797260,2.268493,1,0
2,52.180822,0.616438,0.011814,11.671233,6.934247,2.232877,0,0
3,52.068493,8.326027,0.159905,26.939726,6.676712,1.690411,0,0
4,54.608219,8.323288,0.152418,11.810959,9.473973,3.030137,0,0


### Variables de densité / régions : détecter instabilité géographique

In [ ]:
df_train[["REGION_POPULATION_RELATIVE", "LIVE_REGION_NOT_WORK_REGION", "LIVE_CITY_NOT_WORK_CITY", "REG_REGION_NOT_LIVE_REGION"]].head()

,REGION_POPULATION_RELATIVE,LIVE_REGION_NOT_WORK_REGION,LIVE_CITY_NOT_WORK_CITY,REG_REGION_NOT_LIVE_REGION
0,0.018801,0,0,0
1,0.003541,0,0,0
2,0.010032,0,0,0
3,0.008019,0,0,0
4,0.028663,0,1,0


### Variables documents & flags → somme globale

In [ ]:
df_train[["FLAG_DOCUMENT_2", "FLAG_DOCUMENT_3", "FLAG_DOCUMENT_4"]].head()

,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4
0,0,1,0
1,0,1,0
2,0,0,0
3,0,1,0
4,0,0,0


In [ ]:
# Colonnes : FLAG_DOCUMENT_2 to FLAG_DOCUMENT_21

doc_cols = [col for col in df_train.columns if 'FLAG_DOCUMENT' in col]
df_train['DOCUMENT_COUNT'] = df_train[doc_cols].sum(axis=1)

In [ ]:
df_train["DOCUMENT_COUNT"].value_counts()

DOCUMENT_COUNT
1    270056
0     29549
2      7742
3       163
4         1
Name: count, dtype: int64

### Variables logement / immeuble (AVG / MODE / MEDI)

In [ ]:
df_train[['APARTMENTS_AVG','APARTMENTS_MEDI','APARTMENTS_MODE']].head()

,APARTMENTS_AVG,APARTMENTS_MEDI,APARTMENTS_MODE
0,0.0247,0.0250,0.0252
1,0.0959,0.0968,0.0924
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN


In [ ]:
df_train[[ "BASEMENTAREA_AVG", "BASEMENTAREA_MEDI", "BASEMENTAREA_MODE"]].head()

,BASEMENTAREA_AVG,BASEMENTAREA_MEDI,BASEMENTAREA_MODE
0,0.0369,0.0369,0.0383
1,0.0529,0.0529,0.0538
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN


In [ ]:
df_train[["YEARS_BUILD_AVG", "YEARS_BUILD_MEDI", "YEARS_BUILD_MODE",
          "ELEVATORS_AVG", "ELEVATORS_MEDI", "ELEVATORS_MODE"]].head()

,YEARS_BUILD_AVG,YEARS_BUILD_MEDI,YEARS_BUILD_MODE,ELEVATORS_AVG,ELEVATORS_MEDI,ELEVATORS_MODE
0,0.6192,0.6243,0.6341,0.00,0.00,0.0000
1,0.7960,0.7987,0.8040,0.08,0.08,0.0806
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
client_housing_features = ["APARTMENTS", "BASEMENTAREA", "YEARS_BUILD", "COMMONAREA", "ELEVATORS"]

for item in client_housing_features:
    df_train[f'{item}_MEAN'] = df_train[[f'{item}_AVG', f'{item}_MEDI', f'{item}_MODE']].mean(axis=1)

In [ ]:
df_train.shape

(307511, 268)

In [ ]:
df_train.head()

,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,ID_PUBLISH_YEARS,PHONE_CHANGE_YEARS,RECENT_ID_CHANGE,RECENT_PHONE_CHANGE,DOCUMENT_COUNT,APARTMENTS_MEAN,BASEMENTAREA_MEAN,YEARS_BUILD_MEAN,COMMONAREA_MEAN,ELEVATORS_MEAN
0,100002,1,1.0,0,1,0,202500.0,406597.5,24700.5,351000.0,...,5.808219,3.106849,0,0,1,0.024967,0.037367,0.625867,0.014367,0.0000
1,100003,0,0.0,0,0,0,270000.0,1293502.5,35698.5,1129500.0,...,0.797260,2.268493,1,0,1,0.095033,0.053200,0.799567,0.057000,0.0802
2,100004,0,1.0,1,1,0,67500.0,135000.0,6750.0,135000.0,...,6.934247,2.232877,0,0,0,NaN,NaN,NaN,NaN,NaN
3,100006,0,0.0,0,1,0,135000.0,312682.5,29686.5,297000.0,...,6.676712,1.690411,0,0,1,NaN,NaN,NaN,NaN,NaN
4,100007,0,1.0,0,1,0,121500.0,513000.0,21865.5,513000.0,...,9.473973,3.030137,0,0,1,NaN,NaN,NaN,NaN,NaN


# DATA EXPLORATION APPLICATION

# DATA AGGREGATION

In [ ]:
lst_parquets = [
 'application_test_processed.parquet',
 'application_train_processed.parquet',
 'bureau_agg.parquet',
 'credit_card_agg.parquet',
 'installments_agg.parquet',
 'pos_cash_agg.parquet',
 'previous_agg.parquet'
 ]

In [ ]:
train_df = pd.read_parquet(f'{ROOT_DIR}application_train_processed.parquet')
test_df = pd.read_parquet(f'{ROOT_DIR}application_test_processed.parquet')

In [ ]:
train_only = set(train_df.columns) - set(test_df.columns)
print("Columns only in TRAIN:")
for col in sorted(train_only):
    print(col)

test_only = set(test_df.columns) - set(train_df.columns)
print("\nColumns only in TEST:")
for col in sorted(test_only):
    print(col)

Columns only in TRAIN:
NAME_FAMILY_STATUS_Unknown
NAME_INCOME_TYPE_Maternity leave
TARGET

Columns only in TEST:


In [ ]:
train_df.drop(columns=["NAME_FAMILY_STATUS_Unknown", "NAME_INCOME_TYPE_Maternity leave"], inplace=True)

In [ ]:
# Same features
assert set(train_df.columns) - {'TARGET'} == set(test_df.columns)

print("✅ Train/Test merge OK")

✅ Train/Test merge OK


In [ ]:
bureau_agg = pd.read_parquet(f'{ROOT_DIR}bureau_agg.parquet', engine="fastparquet")
prev_agg = pd.read_parquet(f'{ROOT_DIR}previous_agg.parquet', engine="fastparquet")
pos_agg = pd.read_parquet(f'{ROOT_DIR}pos_cash_agg.parquet', engine="fastparquet")
inst_agg = pd.read_parquet(f'{ROOT_DIR}installments_agg.parquet', engine="fastparquet")
cc_agg = pd.read_parquet(f'{ROOT_DIR}credit_card_agg.parquet', engine="fastparquet")

In [ ]:
for df in [bureau_agg, prev_agg, pos_agg, inst_agg, cc_agg]:
    assert df.index.name == 'SK_ID_CURR', f"Index name should be 'SK_ID_CURR' but got {df.index.name}"

In [ ]:
feature_tables = [
    bureau_agg,
    prev_agg,
    pos_agg,
    inst_agg,
    cc_agg
]

for feat in feature_tables:
    app_train = train_df.join(feat, how='left')
    app_test  = test_df.join(feat, how='left')

In [ ]:
app_train.shape, app_test.shape

((307511, 272), (48744, 271))

In [ ]:
app_train.to_parquet(f'{ROOT_DIR}complete_train.parquet', index=True)
app_test.to_parquet(f'{ROOT_DIR}complete_test.parquet', index=True)